# Fine-tune an LLM with Differential Privacy

This notebook fine-tunes DistilGPT-2 on AG News using LoRA adapters
with DP-SGD. It exercises the full production workflow: adaptive
clipping, truncated Poisson sampling, microbatching, and TorchOpt
optimisation.

**Prerequisites:** [DP-SGD Training](dp_sgd_training.ipynb),
[Sampling & Microbatching](sampling_and_microbatching.ipynb).

**Components exercised:** `make_functional` with `partition_trainable`,
`adaptive_clipped_grad`, `TruncatedPoissonSampler`, `torchopt.adamw`,
`acc.truncated_poisson`, PEFT LoRA.

**Runtime:** ~10--15 minutes on CPU.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchopt
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForCausalLM, AutoTokenizer

import opaque.accounting as acc
from opaque import TruncatedPoissonSampler, gaussian_noise, make_functional
from opaque.clipping import adaptive_clipped_grad
from opaque.random import key

torch.manual_seed(42)
np.random.seed(42)

## Configuration

In [ ]:
MODEL_NAME = "distilgpt2"
LORA_R = 8
LORA_ALPHA = 16

NUM_STEPS = 500
BATCH_SIZE = 32
MAX_BATCH_SIZE = 40
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
MICROBATCH_SIZE = 8
MAX_LENGTH = 128

CLIP_NORM = 0.1
NOISE_MULTIPLIER = 0.24
TARGET_DELTA = 1e-5

LOG_INTERVAL = 50

## Load model and apply LoRA

LoRA reduces the number of trainable parameters to ~0.5% of the full
model, which keeps per-example gradient norms small and makes clipping
more effective.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    target_modules=["c_attn", "c_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Prepare dataset

In [ ]:
dataset = load_dataset("ag_news", split="train")
texts = [ex["text"] for ex in dataset]

tokenized = tokenizer(
    texts,
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

input_ids = tokenized["input_ids"]
attention_mask = tokenized["attention_mask"]
labels = input_ids.clone()
train_dataset = TensorDataset(input_ids, attention_mask, labels)

print(f"Dataset: {len(train_dataset)} examples, sequence length {MAX_LENGTH}")

## Functional conversion and loss function

`partition_trainable=True` separates the LoRA weights (trainable) from
the frozen base model, so only the LoRA gradients are clipped and
noised.

In [ ]:
fmodel, trainable_params, frozen_params = make_functional(
    model,
    disable_autograd_tracking=True,
    partition_trainable=True,
)

trainable_count = sum(p.numel() for p in trainable_params.values())
frozen_count = sum(p.numel() for p in frozen_params.values())
print(f"Trainable: {trainable_count:,}  Frozen: {frozen_count:,}")


def per_example_loss(trainable, frozen, ids, mask, labels):
    all_params = {**frozen, **trainable}
    out = fmodel(
        all_params,
        ids.unsqueeze(0),
        attention_mask=mask.unsqueeze(0),
        labels=labels.unsqueeze(0),
    )
    return out.loss

## Sampler, clipping, noise, and optimizer

In [ ]:
sample_rate = BATCH_SIZE / len(train_dataset)

sampler = TruncatedPoissonSampler(
    train_dataset,
    sample_rate=sample_rate,
    max_batch_size=MAX_BATCH_SIZE,
    num_epochs=NUM_STEPS,
    key=key(42),
)
train_loader = DataLoader(train_dataset, batch_sampler=sampler)

grad_fn, clip_state = adaptive_clipped_grad(
    per_example_loss,
    argnums=0,
    batch_argnums=(2, 3, 4),
    initial_clip_norm=CLIP_NORM,
    target_quantile=0.80,
    microbatch_size=MICROBATCH_SIZE,
    return_aux=True,
    clip_norm_min=0.01,
    clip_norm_max=10.0,
    key=key(0),
)

noise_fn, noise_state = gaussian_noise(
    stddev=NOISE_MULTIPLIER * clip_state.clip_norm,
    key=key(42),
)

optimizer = torchopt.adamw(lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
opt_state = optimizer.init(trainable_params)

# Pre-compute expected final epsilon
expected_eps = (
    acc.truncated_poisson(
        acc.gaussian(NOISE_MULTIPLIER),
        sample_rate=sample_rate,
        batch_size_cap=MAX_BATCH_SIZE,
        dataset_size=len(train_dataset),
    )
    * NUM_STEPS
).epsilon_at(TARGET_DELTA)
print(f"Expected final epsilon: {expected_eps:.2f}")

## Training loop

In [ ]:
losses = []
epsilons = []
clip_norms = []

for step, (batch_ids, batch_mask, batch_labels) in enumerate(train_loader):
    if step >= NUM_STEPS:
        break

    # 1. Clipped gradients (adaptive threshold, microbatched)
    (grads, aux), clip_state = grad_fn(
        trainable_params,
        frozen_params,
        batch_ids,
        batch_mask,
        batch_labels,
        state=clip_state,
    )

    # 2. Add noise scaled to current sensitivity
    stddev = NOISE_MULTIPLIER * clip_state.sensitivity()
    noise_fn, noise_state = gaussian_noise(stddev, key=noise_state.rng_key)
    noisy_grads, noise_state = noise_fn(grads, noise_state)

    # 3. Optimizer step
    updates, opt_state = optimizer.update(
        noisy_grads,
        opt_state,
        params=trainable_params,
    )
    trainable_params = torchopt.apply_updates(trainable_params, updates)

    # 4. Track privacy
    dp_step = acc.truncated_poisson(
        acc.gaussian(NOISE_MULTIPLIER),
        sample_rate=sample_rate,
        batch_size_cap=MAX_BATCH_SIZE,
        dataset_size=len(train_dataset),
    ) * (step + 1)
    epsilon = dp_step.epsilon_at(TARGET_DELTA)

    avg_loss = aux.loss_values.mean().item()
    losses.append(avg_loss)
    epsilons.append(epsilon)
    clip_norms.append(clip_state.clip_norm)

    if (step + 1) % LOG_INTERVAL == 0 or step == 0:
        print(
            f"Step {step + 1:4d}/{NUM_STEPS}  "
            f"loss={avg_loss:.3f}  "
            f"clip={clip_state.clip_norm:.3f}  "
            f"epsilon={epsilon:.2f}"
        )

print(f"\nFinal: loss={losses[-1]:.3f}, epsilon={epsilons[-1]:.2f}")

## Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(losses)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training loss")
axes[0].grid(alpha=0.3)

axes[1].plot(epsilons)
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Epsilon")
axes[1].set_title("Privacy cost")
axes[1].grid(alpha=0.3)

axes[2].plot(clip_norms)
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Clip norm")
axes[2].set_title("Adaptive clip norm")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

Key choices for LLM fine-tuning with DP:

- **LoRA** reduces trainable parameters to <1% of the model, keeping
  gradient norms small.
- **Adaptive clipping** (`adaptive_clipped_grad`) avoids manual tuning
  of the clip threshold.
- **Truncated Poisson sampling** bounds memory while preserving privacy
  amplification.
- **Microbatching** (`microbatch_size`) further reduces peak memory.
- **`partition_trainable`** ensures only LoRA weights are clipped and
  noised.

For hyperparameter guidance see the
[HuggingFace Compatibility guide](../user-guide/huggingface.md).